# Classifier transfor leaning from LLM 情緒類別分類器(遷移學習)

This is a comprahensive notebook and tutorial on how to fine tune the `qwen-0.5b` classification model

Fine-tuning Qwen-0.5B (a smaller model) with LoRA (Low-Rank Adaptation) is an efficient approach that requires less computational power.

It got a classification accuracy of 0.93.


# Colab GPU
## 設定執行階段 變更執行階段類型 硬體加速器  選取GPU  


In [1]:
!nvidia-smi

Thu Apr  3 20:49:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 561.09                 Driver Version: 561.09         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090      WDDM  |   00000000:01:00.0 Off |                  Off |
| 46%   69C    P2            407W /  450W |    2818MiB /  24564MiB |     87%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 列出GPU與CPU資源
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 16644549555763398029
xla_global_id: -1
]


In [3]:
# Specify the GPU device to use
#import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Ubuntu版本資訊

In [4]:
!lsb_release -a

'lsb_release' ���O�����Υ~���R�O�B�i���檺�{���Χ妸�ɡC


# 掛載雲端硬碟

In [5]:
# from google.colab import drive
# drive.mount('/content/drive')

# 切換工作目錄到雲端硬碟目錄下(取用資料比較方便，可以用相對路徑)

你要改成你自己雲端硬碟目錄的路徑，可複製路徑並貼上，不要用鍵盤輸入!

In [6]:
# cd to_your_folder
# cd命令的前面一行不要加上說明文字，否則colab的cd會認不得指令

# Insatll packages


        transformers: For model handling.
        peft: For LoRA integration.
        datasets: For dataset handling.
        accelerate: For distributed training.
        bitsandbytes: For memory-efficient training (optional but useful for Qwen).

In [7]:
#!pip install transformers peft datasets accelerate bitsandbytes

In [ ]:
import torch
import datasets
import pandas as pd
import evaluate
import numpy as np

# Load Huggingface transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer
from transformers import BertTokenizer, BertTokenizerFast, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn import metrics
import torch

In [9]:
# Setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


# Read data

簡體中文資料，資料集來自於網路，轉換成繁體中文

In [10]:
df = pd.read_csv('./dataset_reviews.csv', sep='|')

In [11]:
df

,text,label
0,做父母一定要有劉墉這樣的心態，不斷地學習，不斷地進步，不斷地給自己補充新鮮血液，讓自己保持一...,1
1,作者真有英國人嚴謹的風格，提出觀點、進行論述論證，儘管本人對物理學瞭解不深，但是仍然能感受到...,1
2,作者長篇大論借用詳細報告數據處理工作和計算結果支持其新觀點。為什麼荷蘭曾經縣有歐洲最高的生產...,1
3,作者在戰幾時之前用了〞擁抱〞令人叫絕．日本如果沒有戰敗，就有會有美軍的佔領，沒胡官僚主義的延...,1
4,作者在少年時即喜閱讀，能看出他精讀了無數經典，因而他有一個龐大的內心世界。他的作品最難能可貴...,1
...,...,...
80437,以前幾乎天天吃，現在調料什麼都不放，,0
80438,昨天訂涼皮兩份，什麼調料都沒有放，就放了點麻油，特別難吃，丟了一份，再也不想吃了,0
80439,"涼皮太辣,吃不下都",0
80440,本來遲到了還自己點！！！,0


In [12]:
df.dtypes

text     object
label     int64
dtype: object

# Convert the format of y 

Convert the format of y from int to LongTensor

## Convert label using one-hot representation 輸出資料格式one-hot轉換
    
    轉成用2個節點表達兩類
    類別0: [1 0]  
    類別1: [0 1]  


    如果是3個類別用3個節點表之:
    類別0: [1 0 0]  
    類別1: [0 1 0]  
    類別2: [0 0 1]

負面情緒Negative 0 --> [1 0] 

正面情緒Positive 1 --> [0 1] 

## Easy Represention 0,1,2... (內部會自動轉換為one-hot)

    負面情緒Negative 0 --> [0] 

    正面情緒Positive 1 --> [1] 

    如果是3個類別
    中立情緒 Neutral 2 --> [2] 


In [13]:
# Map labels to integers
categories=['負面','正面']

In [14]:

label_to_id = { cate : i for i, cate in enumerate(categories)}

In [15]:
label_to_id

{'負面': 0, '正面': 1}

In [16]:
id_to_label = { i : cate for i, cate in enumerate(categories)}

In [17]:
id_to_label

{0: '負面', 1: '正面'}

# Sample some examples for demonstration

In [ ]:
df = df.sample(20000)

# Conver pandas dataframe to Huggingface Dataset

In [19]:
dataset = datasets.Dataset.from_pandas(df, preserve_index=False)
# eval_data = Dataset.from_pandas(X_eval)

In [20]:
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 80442
})

In [21]:
dataset[0]

{'text': '做父母一定要有劉墉這樣的心態，不斷地學習，不斷地進步，不斷地給自己補充新鮮血液，讓自己保持一顆年輕的心。我想，這是他能很好的和孩子溝通的一個重要因素。讀劉墉的文章，總能讓我看到一個快樂的平易近人的父親，他始終站在和孩子同樣的高度，給孩子創造著一個充滿愛和自由的生活環境。很喜歡劉墉在字裡行間流露出的做父母的那種小狡黠，讓人總是忍俊不禁，父母和子女之間有時候也是一種戰鬥，武力爭鬥過於低級了，智力較量才更有趣味。所以，做父母的得加把勁了，老思想老觀念注定會一敗塗地，生命不息，學習不止。家庭教育，真的是樂在其中。',
 'label': 1}

In [22]:
dataset.to_pandas().head(5)

,text,label
0,做父母一定要有劉墉這樣的心態，不斷地學習，不斷地進步，不斷地給自己補充新鮮血液，讓自己保持一...,1
1,作者真有英國人嚴謹的風格，提出觀點、進行論述論證，儘管本人對物理學瞭解不深，但是仍然能感受到...,1
2,作者長篇大論借用詳細報告數據處理工作和計算結果支持其新觀點。為什麼荷蘭曾經縣有歐洲最高的生產...,1
3,作者在戰幾時之前用了〞擁抱〞令人叫絕．日本如果沒有戰敗，就有會有美軍的佔領，沒胡官僚主義的延...,1
4,作者在少年時即喜閱讀，能看出他精讀了無數經典，因而他有一個龐大的內心世界。他的作品最難能可貴...,1


# Load Tokenizer

In [23]:

# model_id = "google/gemma-3-1b"
# model_id = "Qwen/Qwen2.5-0.5B"
model_id = "Qwen/Qwen2.5-0.5B-instruct"

In [24]:
tokenizer = AutoTokenizer.from_pretrained(model_id)


In [25]:
# 必須在Huggingface註冊，取得API token才能下載模型
#access_token = "???"
#tokenizer = AutoTokenizer.from_pretrained(model_id, token=access_token)

# Tokenzie text

In [26]:
def tokenize_function(example):
    return tokenizer(
        example["text"], 
        #padding="max_length", 
        max_length=512,
        truncation=True, 
    )

#tokenized_dataset = dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/80442 [00:00<?, ? examples/s]

In [27]:
tokenized_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 80442
})

In [28]:
# tokenized_dataset[0]

# Split dataset for training and testing

# Split dataset: Train, Test (Val) 

Training set: 訓練資料集 -->給模型讀進去訓練

Test set: 測試資料集 -->驗證或測試模型的準確度



Split dataset into 90% for training and 10% for testing

In [29]:
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.05, seed=1234)
train_data = tokenized_dataset["train"]
test_data = tokenized_dataset["test"]

In [30]:
train_data

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 76419
})

In [31]:
print(test_data)

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 4023
})


In [32]:
train_data[0]

{'text': '不滿意，色差太大！做工一般！！',
 'label': 0,
 'input_ids': [16530,
  101496,
  36589,
  3837,
  38035,
  99572,
  102791,
  6313,
  115238,
  100141,
  31251],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

#  定義Model

這裡的做法有簡單的版本也有複雜的版本

簡單版:


        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,  # 模型名稱
            num_labels=len(categories),  # Number of output labels
        )


複雜版:




In [33]:
from transformers import Qwen2Model, Trainer, TrainingArguments, AutoTokenizer, AutoModel
from torch import nn
from transformers.modeling_outputs import SequenceClassifierOutput
import os

In [ ]:
import torch
import os
import torch.nn.functional as F
from torch import nn

class QwenForClassifier(nn.Module):
    def __init__(self, base_model, hidden_size, num_labels):
        super(QwenForClassifier, self).__init__()
        # 凍結 base model 的參數
        self.base_model = base_model
        
        for param in self.base_model.parameters():
            param.requires_grad = False
            
        # 注意力池化機制
        self.attention_pooler = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        # 多層融合權重 (最後4層)
        self.layer_weights = nn.Parameter(torch.ones(4) / 4)
        
        # 增強型分類器
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),
            
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            
            nn.Linear(128, num_labels)
        )
        
        # 保存配置
        self.config = base_model.config
        self.config.num_labels = num_labels
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        # 獲取所有隱藏層狀態
        outputs = self.base_model(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        # 獲取最後4層隱藏狀態
        hidden_states = outputs.hidden_states
        if hidden_states is None:
            # 如果模型沒有返回hidden_states，使用last_hidden_state
            last_hidden = outputs.last_hidden_state
            sequence_output = last_hidden
        else:
            # 融合最後4層 (或可用層數)
            last_layers = hidden_states[-4:] if len(hidden_states) >= 4 else hidden_states[1:]
            layer_weights = F.softmax(self.layer_weights[:len(last_layers)], dim=0)
            
            # 加權融合多層特徵
            sequence_output = torch.zeros_like(last_layers[0])
            for i, layer in enumerate(last_layers):
                sequence_output += layer_weights[i].unsqueeze(-1).unsqueeze(-1) * layer
        
        # 注意力池化
        attention_scores = self.attention_pooler(sequence_output)
        attention_probs = F.softmax(attention_scores, dim=1)
        context_vector = torch.matmul(attention_probs.transpose(-1, -2), sequence_output).squeeze(1)
        
        # 也計算平均池化向量
        mean_pooled = torch.mean(sequence_output, dim=1)
        
        # 結合注意力池化和平均池化 (殘差連接)
        combined_repr = context_vector + mean_pooled
            
        # 分類預測
        logits = self.classifier(combined_repr)
        
        # 計算損失
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
            
        return {"loss": loss, "logits": logits}
    
    def save_model(self, output_dir=None):
        """保存分類器權重和配置"""
        os.makedirs(output_dir, exist_ok=True)
        
        # 保存分類器權重
        classifier_path = os.path.join(output_dir, "classifier_weights.pt")
        model_dict = {
            'classifier': self.classifier.state_dict(),
            'attention_pooler': self.attention_pooler.state_dict(),
            'layer_weights': self.layer_weights,
            'config': {
                'num_labels': self.config.num_labels,
                'hidden_size': self.config.hidden_size
            }
        }
        torch.save(model_dict, classifier_path)
        print(f"已保存分類器權重至 {classifier_path}")
    
    def load_model(self, model_dir, device=None):
        """載入分類器權重"""
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            
        classifier_path = os.path.join(model_dir, "classifier_weights.pt")
        if os.path.exists(classifier_path):
            model_dict = torch.load(classifier_path, map_location=device, weights_only=True)
            
            # 載入各組件
            self.classifier.load_state_dict(model_dict['classifier'])
            self.attention_pooler.load_state_dict(model_dict['attention_pooler'])
            self.layer_weights.data = model_dict['layer_weights'].to(device)
            
            print(f"已載入分類器權重: {classifier_path}")
            return True
        else:
            print(f"警告: 找不到分類器權重檔案 {classifier_path}")
            return False

## 初始化模型


        分類任務：兩者都可用，但 AutoModel 更輕量
        生成功能：只有 AutoModelForCausalLM 支持
        內存使用：AutoModelForCausalLM 通常較大，因為包含了完整的語言模型頭
        在 Qwen2 情感分類模型中，使用 AutoModelForCausalLM 更為靈活，因為它既可以進行分類，也保留了原始的文本生成功能。



        # AutoModelForCausalLM 輸出
        outputs = base_model(input_ids, attention_mask)
        # 輸出包含：
        # - loss: (可選) 語言模型損失
        # - logits: 張量，形狀為 [batch_size, sequence_length, vocab_size]
        # - past_key_values: (可選) 用於加速解碼的過去狀態
        # - hidden_states: (可選) 所有隱藏層狀態的元組
        # - attentions: (可選) 注意力權重的元組


        # AutoModel 輸出
        outputs = base_model(input_ids, attention_mask)
        # 輸出包含：
        # - last_hidden_state: 張量，形狀為 [batch_size, sequence_length, hidden_size]
        # - hidden_states: (可選) 所有隱藏層狀態的元組
        # - attentions: (可選) 注意力權重的元組

In [36]:
len(categories)

2

In [37]:
# 在外部先載入base_model預訓練權重(不包含分類層)
full_model = AutoModelForCausalLM.from_pretrained(model_id)
#
hidden_size = full_model.config.hidden_size
model = QwenForClassifier(full_model.model, hidden_size, num_labels= len(categories))

# 移動到指定設備
model = model.to(device)

In [38]:
full_model.model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2SdpaAttention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
        (rotary_emb): Qwen2RotaryEmbedding()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

In [39]:
full_model.base_model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2SdpaAttention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
        (rotary_emb): Qwen2RotaryEmbedding()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

## 看看base_model與model有何不同?

model內部有: 一個base_model+輸出分類層。它被拼接為分類器，可以做兩個類別的分類

base_model仍舊是GPT自回歸模型

但是在記憶體，兩者是共享同一份base_model權重，model有多一個分類器的輸出層的部分

In [40]:
full_model 

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

In [41]:
model

QwenForClassifier(
  (base_model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMS

In [42]:
# 這與model是一樣的模型??
model.base_model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2SdpaAttention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
        (rotary_emb): Qwen2RotaryEmbedding()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

# 模型怎麼用?

尚未訓練的模型 (classifier的數據都是0)

In [43]:

# Function to make predictions
def predict_sentiment(text, model, tokenizer, device):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract logits and apply softmax to get probabilities
    # logits = outputs.logits
    logits = outputs["logits"]  # 取出 logits
    
    
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    
    # Get the predicted class (0: negative, 1: positive)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    
    # Get the class name using id_to_label
    predicted_label = id_to_label[predicted_class]
    
    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()
    
    return {
        "text": text,
        "sentiment": predicted_label,
        "confidence": round(confidence,2),
        "probabilities": {
            id_to_label[i]: round(prob.item(),2) for i, prob in enumerate(probabilities[0])
        }
    }


In [44]:
text = "今天天氣真好，我很開心"
predict_sentiment(text, model, tokenizer, device)

{'text': '今天天氣真好，我很開心',
 'sentiment': '正面',
 'confidence': 0.6,
 'probabilities': {'負面': 0.4, '正面': 0.6}}

In [45]:
text = "這個產品品質差，服務更糟糕"
predict_sentiment(text, model, tokenizer, device)

{'text': '這個產品品質差，服務更糟糕',
 'sentiment': '正面',
 'confidence': 0.6,
 'probabilities': {'負面': 0.4, '正面': 0.6}}

In [46]:
text = "這家餐廳的食物美味，環境也很舒適"
predict_sentiment(text, model, tokenizer, device)

{'text': '這家餐廳的食物美味，環境也很舒適',
 'sentiment': '正面',
 'confidence': 0.53,
 'probabilities': {'負面': 0.47, '正面': 0.53}}

## 模型每個參數層都是Trainable

In [47]:
def print_model_details(model):
    print("\n" + "="*80)
    print("MODEL ARCHITECTURE WITH TRAINABILITY STATUS")
    print("="*80)
    
    trainable_params = 0
    non_trainable_params = 0
    
    # Print each layer with details
    for idx, (name, layer) in enumerate(model.named_modules(), 1):
        if not list(layer.named_children()):  # Only print leaf nodes (actual layers)
            param_count = sum(p.numel() for p in layer.parameters())
            status = "TRAINABLE" if any(p.requires_grad for p in layer.parameters()) else "NON-TRAINABLE"
            
            print(f"\nLayer #{idx:02d}")
            print(f"├─ Name: {name}")
            print(f"├─ Type: {layer.__class__.__name__}")
            print(f"├─ Details: {layer}")
            print(f"├─ Parameters: {param_count:,}")
            print(f"└─ Status: {status}")
            
            if status == "TRAINABLE":
                trainable_params += param_count
            else:
                non_trainable_params += param_count
    
    total_params = trainable_params + non_trainable_params
    trainable_percentage = (trainable_params / total_params) * 100 if total_params > 0 else 0
    
    print("\n" + "="*80)
    print(f"Total Trainable Parameters: {trainable_params:,}")
    print(f"Total Non-Trainable Parameters: {non_trainable_params:,}")
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters Percentage: {trainable_percentage:.2f}%")
    print("="*80)

In [48]:
print_model_details(model)


MODEL ARCHITECTURE WITH TRAINABILITY STATUS

Layer #03
├─ Name: base_model.embed_tokens
├─ Type: Embedding
├─ Details: Embedding(151936, 896)
├─ Parameters: 136,134,656
└─ Status: NON-TRAINABLE

Layer #07
├─ Name: base_model.layers.0.self_attn.q_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=896, bias=True)
├─ Parameters: 803,712
└─ Status: NON-TRAINABLE

Layer #08
├─ Name: base_model.layers.0.self_attn.k_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=128, bias=True)
├─ Parameters: 114,816
└─ Status: NON-TRAINABLE

Layer #09
├─ Name: base_model.layers.0.self_attn.v_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=128, bias=True)
├─ Parameters: 114,816
└─ Status: NON-TRAINABLE

Layer #10
├─ Name: base_model.layers.0.self_attn.o_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=896, bias=False)
├─ Parameters: 802,816
└─ Status: NON-TRAINABLE

Layer #11
├─ Name: base_model.layers.0.self_attn.rotary_emb

In [49]:
# 檢查原始參數是否可訓練
# print("Classifier weight requires_grad:", model.classifier.weight.requires_grad)

## 模型主體固定參數，只有極少數參數是Trainable

In [50]:
#model.print_trainable_parameters()

In [51]:
def print_trainable_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable_params:,d} || all params: {all_params:,d} || trainable%: {100 * trainable_params / all_params:.2f}%")

In [52]:
print_trainable_parameters(model)

trainable params: 378,503 || all params: 494,411,271 || trainable%: 0.08%


In [53]:
print_model_details(model)


MODEL ARCHITECTURE WITH TRAINABILITY STATUS

Layer #03
├─ Name: base_model.embed_tokens
├─ Type: Embedding
├─ Details: Embedding(151936, 896)
├─ Parameters: 136,134,656
└─ Status: NON-TRAINABLE

Layer #07
├─ Name: base_model.layers.0.self_attn.q_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=896, bias=True)
├─ Parameters: 803,712
└─ Status: NON-TRAINABLE

Layer #08
├─ Name: base_model.layers.0.self_attn.k_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=128, bias=True)
├─ Parameters: 114,816
└─ Status: NON-TRAINABLE

Layer #09
├─ Name: base_model.layers.0.self_attn.v_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=128, bias=True)
├─ Parameters: 114,816
└─ Status: NON-TRAINABLE

Layer #10
├─ Name: base_model.layers.0.self_attn.o_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=896, bias=False)
├─ Parameters: 802,816
└─ Status: NON-TRAINABLE

Layer #11
├─ Name: base_model.layers.0.self_attn.rotary_emb

## Use EOS token as padding

In [54]:
tokenizer.pad_token

'<|endoftext|>'

In [55]:
tokenizer.eos_token

'<|im_end|>'

In [56]:
model.pad_token_id = tokenizer.eos_token_id  # Use EOS token as padding

# Train

In [ ]:
      
class CustomTrainer(Trainer):
    def save_model(self, output_dir=None, _internal_call=False):
        """保存分類器權重和配置"""
        # 使用預設output_dir如果未指定
        output_dir = output_dir if output_dir is not None else self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)
        
        # 保存分類器權重
        classifier_path = os.path.join(output_dir, "classifier_weights.pt")
        model_dict = {
            'classifier': self.model.classifier.state_dict(),
            'attention_pooler': self.model.attention_pooler.state_dict(), 
            'layer_weights': self.model.layer_weights,
            'config': {
                'num_labels': self.model.config.num_labels,
                'hidden_size': self.model.config.hidden_size
            }
        }
        
        # Save model config (required by HF Trainer)
        if hasattr(self.model, "config"):
            self.model.config.save_pretrained(output_dir)
        
        torch.save(model_dict, classifier_path)
        print(f"已保存分類器權重至 {classifier_path}")
        
        # 保存訓練狀態
        super().save_state()
        
        return output_dir
    
  

In [58]:
#metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])
#metric = evaluate.combine(["accuracy"])
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)  # Convert probabilities to predicted labels
    metric = evaluate.load('accuracy')
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# requries several GB of GPU memory
training_args = TrainingArguments(
    output_dir="checkpoints_v4",  # Output directory for checkpoints
    learning_rate=5e-5,  # Learning rate for the optimizer
    weight_decay=0.01,  # Weight decay for regularization
    warmup_steps=500,
    seed=101,
    
    per_device_train_batch_size=26,  # Batch size per device
    per_device_eval_batch_size=3,  # Batch size per device for evaluation 
    
    num_train_epochs=5,  # Number of training epochs
    
    eval_strategy='steps',  # Evaluate after each epoch
    save_strategy="steps",  # Save model checkpoints after each epoch
    
    #load_best_model_at_end=True,  # Load the best model based on the chosen metric
    save_total_limit=2,
    push_to_hub=False,  # Disable pushing the model to the Hugging Face Hub 
    report_to="none",  # Disable logging to Weight&Bias
    #fp16=True, # 是否用此精度訓練Whether to use fp16 16-bit (mixed) precision training instead of 32-bit training.
    #bf16=True, # for 新型GPU 才能設定bf16
    logging_steps=1000,
)  

In [60]:
trainer = CustomTrainer(
    model=model,  # The LoRA-adapted model
    #tokenizer=tokenizer,  # Tokenizer for the model
    processing_class=tokenizer,  # Tokenizer for the model
    
    train_dataset=train_data,  # Training dataset
    eval_dataset=test_data,  # Evaluation dataset
    args=training_args,  # Training arguments
    compute_metrics=compute_metrics,  # Function to calculate evaluation metrics
)

## Resume training from the last checkpoint if available

接續訓練

訓練後，手動載入之前訓練的分類器權重

In [ ]:

# checkpoint_path = "./checkpoints_v3/checkpoint-3167"
# model_path = "trained_classifier_v3"
# model.load_model(checkpoint_path)

## Let's train the model

In [62]:
%%time
model.config.use_cache = False  # Disable cache for faster training
trainer.train()
# trainer.train(resume_from_checkpoint=True) # Resume training from a checkpoint
#trainer.train(resume_from_checkpoint="./checkpoints_v1/checkpoint-19002")

  0%|          | 0/63685 [00:00<?, ?it/s]

Saved checkpoint classifier weights to checkpoints_v4\checkpoint-500\classifier_weights.pt
{'loss': 0.4608, 'grad_norm': 18.343847274780273, 'learning_rate': 1.9685954306351575e-05, 'epoch': 0.08}


  0%|          | 0/4023 [00:00<?, ?it/s]

{'eval_loss': 0.2910600006580353, 'eval_accuracy': 0.8841660452398707, 'eval_runtime': 79.765, 'eval_samples_per_second': 50.436, 'eval_steps_per_second': 50.436, 'epoch': 0.08}
Saved checkpoint classifier weights to checkpoints_v4\checkpoint-1000\classifier_weights.pt
Saved checkpoint classifier weights to checkpoints_v4\checkpoint-1500\classifier_weights.pt
{'loss': 0.296, 'grad_norm': 8.61012077331543, 'learning_rate': 1.937190861270315e-05, 'epoch': 0.16}


  0%|          | 0/4023 [00:00<?, ?it/s]

...

  0%|          | 0/4023 [00:00<?, ?it/s]

{'eval_loss': 0.27217984199523926, 'eval_accuracy': 0.9316430524484216, 'eval_runtime': 71.1728, 'eval_samples_per_second': 56.524, 'eval_steps_per_second': 56.524, 'epoch': 4.87}
Saved checkpoint classifier weights to checkpoints_v4\checkpoint-62000\classifier_weights.pt
Saved checkpoint classifier weights to checkpoints_v4\checkpoint-62500\classifier_weights.pt
{'loss': 0.2899, 'grad_norm': 4.636594295501709, 'learning_rate': 2.151213001491717e-07, 'epoch': 4.95}


  0%|          | 0/4023 [00:00<?, ?it/s]

{'eval_loss': 0.2733895480632782, 'eval_accuracy': 0.930648769574944, 'eval_runtime': 68.5292, 'eval_samples_per_second': 58.705, 'eval_steps_per_second': 58.705, 'epoch': 4.95}
Saved checkpoint classifier weights to checkpoints_v4\checkpoint-63000\classifier_weights.pt
Saved checkpoint classifier weights to checkpoints_v4\checkpoint-63500\classifier_weights.pt
Saved checkpoint classifier weights to checkpoints_v4\checkpoint-63685\classifier_weights.pt
{'train_runtime': 9009.4264, 'train_samples_per_second': 42.411, 'train_steps_per_second': 7.069, 'train_loss': 0.2868387020279102, 'epoch': 5.0}
CPU times: total: 2h 22min 12s
Wall time: 2h 30min 9s


TrainOutput(global_step=63685, training_loss=0.2868387020279102, metrics={'train_runtime': 9009.4264, 'train_samples_per_second': 42.411, 'train_steps_per_second': 7.069, 'total_flos': 0.0, 'train_loss': 0.2868387020279102, 'epoch': 5.0})

# Make a test

In [63]:
text = "今天天氣真好，我很開心"
predict_sentiment(text, model, tokenizer, device)

{'text': '今天天氣真好，我很開心',
 'sentiment': '正面',
 'confidence': 1.0,
 'probabilities': {'負面': 0.0, '正面': 1.0}}

In [64]:
text = "這個產品品質差，服務更糟糕"
predict_sentiment(text, model, tokenizer, device)

{'text': '這個產品品質差，服務更糟糕',
 'sentiment': '負面',
 'confidence': 1.0,
 'probabilities': {'負面': 1.0, '正面': 0.0}}

# Save Model

In [65]:
#存lora_model
# model_path = "trained_lora_model_v1"
# trainer.model.save_pretrained(model_path)

In [66]:
# 這樣存檔 - 只會儲存分類器權重和設定
# model_path = "trained_classifier_v4"
# trainer.save_model(model_path)
# model.save_pretrained(model_path)

In [67]:
# 這樣存檔 - 只會儲存分類器權重和設定
model_path = "trained_classifier_v4"
model.save_model(model_path)
# model.save_pretrained(model_path)

已保存分類器權重至 trained_classifier_v4\classifier_weights.pt


# Evaluate the model

In [68]:
def predict(input_text):
    inputs = tokenizer(input_text, return_tensors="pt").to(device)  # Convert to PyTorch tensors and move to GPU (if available)
    with torch.no_grad():
        #outputs = model(**inputs).logits  # Get the model's output logits
        outputs = model(**inputs)['logits']  # Get the model's output logits
        y_prob = torch.sigmoid(outputs).tolist()[0]  # Apply sigmoid activation and convert to list
    return np.round(y_prob, 2)  # Round the predicted probability to 2 decimal places

In [69]:
def inference(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs['logits'], dim=1)
    #probs = torch.softmax(outputs.logits, dim=1)
    return probs.argmax().item()

In [70]:
df_test = pd.DataFrame(data=test_data)

In [71]:
df_test

,text,label,input_ids,attention_mask
0,顏色比想像的要深得多……感謝快遞員，每次都非常好,1,"[119283, 38035, 56006, 111176, 9370, 30534, 99...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,虧了，前一天買的108，第二天看就成99了…這也太,0,"[83164, 100, 34187, 3837, 115982, 102810, 9370...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,太大的遊戲有點卡 其他的都很好 看電視特別爽,1,"[99222, 99750, 110256, 114863, 99603, 34369, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
3,是正品哦，套裝系列的，買得很划算，用了小樣，不怎麼好拍照，真的很划算。,1,"[20412, 116470, 104170, 3837, 99619, 105290, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,一個涼菜十塊錢，就退了七塊五，打電話過去，說那邊很忙，五分鐘以後回過來電話，也沒信了！太沒誠...,0,"[104061, 118178, 99800, 94498, 111145, 102666,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
...,...,...,...,...
4018,非常差穿一回不想穿二回,0,"[99491, 99572, 99621, 14777, 18397, 104359, 99...","[1, 1, 1, 1, 1, 1, 1, 1, 1]"
4019,收到的火龍果是爛的，上面寫的是冷鏈收到卻是爛的,0,"[104381, 9370, 79599, 106430, 27773, 20412, 75...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4020,歐式簡約裝潢，有飄窗，可惜周圍全是樓，大床是2張小床拼成的，可以看出小床很小，羽絨被、枕頭，...,1,"[112311, 28330, 103271, 104894, 105290, 121501...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4021,包裝的很結實，一直信賴沙宣，信賴京東。看保質期都到2020年，很好。,1,"[67279, 105290, 9370, 99165, 100331, 99862, 38...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [72]:
input_text='''立法院今天發布人事通報，總務處長周傑調任參事室參事並代理總務處長，引發外界聯想立法院長韓國瑜出手進行人事調整。周傑今天受訪表示，正常調動，謝謝關心。
部分媒體報導，質疑立法院總務處長人事異動，是否與先前立法院立委辦公室傳出有裝潢費昂貴、不合理等情形，或是日前立法院司法及法制委員會有委員提議要更改格局，而周傑回應「建議要更動是否可在會期中研議規畫，休會時再改」，令國民黨立法院黨團總召傅崐萁不以為然有關。'''
predict(input_text)

array([0.13, 0.85])

In [73]:
input_text = "我不高興"
inference(input_text)

0

In [86]:
%%time
df_test['prediction'] = df_test['text'].map(predict)
df_test['y_pred'] = df_test['prediction'].apply(lambda x: np.argmax(x, axis=0)) 

# 
# df_test['prediction'] = df_test['text'].map(inference)

CPU times: total: 1min 2s
Wall time: 1min 5s


In [87]:
accuracy = (df_test['y_pred'] == df_test['label']).mean()
print(f"Model Accuracy on Test Data: {accuracy:.4f}")
df_test.head()

Model Accuracy on Test Data: 0.9306


,text,label,input_ids,attention_mask,prediction,y_pred
0,顏色比想像的要深得多……感謝快遞員，每次都非常好,1,"[119283, 38035, 56006, 111176, 9370, 30534, 99...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.06, 0.92]",1
1,虧了，前一天買的108，第二天看就成99了…這也太,0,"[83164, 100, 34187, 3837, 115982, 102810, 9370...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.95, 0.07]",0
2,太大的遊戲有點卡 其他的都很好 看電視特別爽,1,"[99222, 99750, 110256, 114863, 99603, 34369, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]","[0.06, 0.88]",1
3,是正品哦，套裝系列的，買得很划算，用了小樣，不怎麼好拍照，真的很划算。,1,"[20412, 116470, 104170, 3837, 99619, 105290, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.04, 0.95]",1
4,一個涼菜十塊錢，就退了七塊五，打電話過去，說那邊很忙，五分鐘以後回過來電話，也沒信了！太沒誠...,0,"[104061, 118178, 99800, 94498, 111145, 102666,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.96, 0.06]",0


In [89]:
cm = metrics.confusion_matrix(df_test['label'], df_test['y_pred'])
cm

array([[1962,  104],
       [ 175, 1782]], dtype=int64)

In [90]:
print(metrics.classification_report(df_test['label'], df_test['y_pred']))

              precision    recall  f1-score   support

           0       0.92      0.95      0.93      2066
           1       0.94      0.91      0.93      1957

    accuracy                           0.93      4023
   macro avg       0.93      0.93      0.93      4023
weighted avg       0.93      0.93      0.93      4023



In [79]:
metrics.precision_score(df_test['label'], df_test['y_pred'], average='micro')

0.9281630623912503

In [80]:
metrics.precision_score(df_test['label'], df_test['y_pred'], average='macro')

0.9291980777462081

In [81]:
metrics.recall_score(df_test['label'], df_test['y_pred'], average='micro')

0.9281630623912503

In [82]:
metrics.recall_score(df_test['label'], df_test['y_pred'], average='macro')

0.9275643667011118

# Load model

In [ ]:
# 在外部先載入base_model預訓練權重(不包含分類層)
# full_model = AutoModelForCausalLM.from_pretrained(model_id)
#
model_path = "trained_classifier_v4"
# model_path = "checkpoints_v3\checkpoint-4145"
hidden_size = full_model.config.hidden_size
model = QwenForClassifier(full_model.model, hidden_size, num_labels= len(categories))

model.load_model(model_path, device=device)

# 移動到指定設備
model = model.to(device)

已載入分類器權重: trained_classifier_v3\classifier_weights.pt


In [ ]:
text = "這個產品品質差，服務更糟糕"
predict_sentiment(text, model, tokenizer, device)

{'text': '這個產品品質差，服務更糟糕',
 'sentiment': '正面',
 'confidence': 0.55,
 'probabilities': {'負面': 0.45, '正面': 0.55}}

In [ ]:
full_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

# Text generation fro full_model

In [ ]:
from IPython.display import Markdown

In [ ]:
def generate_text(input_prompt):
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": input_prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    prompt_length = model_inputs['input_ids'].shape[1]

    generated_ids = full_model.generate(
        model_inputs.input_ids,
        max_new_tokens=512,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(generated_ids[0][prompt_length:], skip_special_tokens=True)
    return response


In [ ]:
text="給出三個保持健康的提示。"
result = generate_text(text)
Markdown(result)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


1. 保持充足的睡眠：良好的睡眠对于身体和心理健康至关重要，建议成年人每晚至少睡7-9小时。

2. 均衡饮食：摄取均衡的营养，包括足够的蛋白质、碳水化合物、脂肪、维生素和矿物质。避免过多摄入加工食品和高糖食品。

3. 定期锻炼：定期进行有氧运动和力量训练可以帮助增强心肺功能，提高免疫力，并减轻压力。建议每周至少进行150分钟的中等强度运动或75分钟的高强度运动。

In [ ]:
%%time
text="我們如何減少空氣污染？請給幾項重要的建議。"
result = generate_text(text)
Markdown(result)

CPU times: total: 2.92 s
Wall time: 2.99 s


1. 設計和建造低排放建築物，使用可再生能源。
2. 運用更高效的交通工具，例如公車、電力車、公共交通工具等。
3. 改進能源消費方式，如使用节能电器和灯具。
4. 提高能效的產品和技術，以降低能源消耗。
5. 篩選和回收廢棄物，減少垃圾填埋和焚烧。
6. 限制工业废气排放，採用高效能的工業設備和技術。
7. 增加綠色能源的使用比例，提高能源效率。
8. 推廣低碳生活方式，如少用一次性塑料制品、節約用水用电等。
9. 加強環境保護法律和政策，促進企業遵守环保要求。
10. 加强環境教育和宣傳，提高公民的環境意識。